### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import glob
import warnings

### Import and Merge files

In [2]:
files = glob.glob(r"C:\Users\ragha\OneDrive\Desktop\Projects\Cyclistic_Bike_Analysis\*.csv")

df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

In [3]:
df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,5C00A6B5A60E4AFC,electric_bike,2025-08-28 15:56:46.803,2025-08-28 16:11:54.890,NaN,NaN,NaN,NaN,41.880000,-87.640000,41.89,-87.62,casual
1,FB25D4B923656D01,electric_bike,2025-08-27 19:15:53.059,2025-08-27 19:34:13.683,NaN,NaN,NaN,NaN,41.950000,-87.640000,41.89,-87.61,casual
2,C665B0EC937C860C,electric_bike,2025-08-27 20:27:31.394,2025-08-27 20:38:29.890,NaN,NaN,NaN,NaN,41.920000,-87.650000,41.90,-87.64,casual
3,C68C54D5FD9717C2,electric_bike,2025-08-28 15:33:21.055,2025-08-28 15:39:58.052,NaN,NaN,NaN,NaN,41.890000,-87.650000,41.88,-87.65,casual
4,902D69C29B0E65F8,electric_bike,2025-08-28 09:59:44.846,2025-08-28 10:11:17.343,NaN,NaN,NaN,NaN,41.980000,-87.680000,41.95,-87.70,casual
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6037963,D5CD1C4E7B56E481,electric_bike,2026-07-05 12:29:45.640,2026-07-05 12:35:02.419,May St & Taylor St,CHI00322,NaN,NaN,41.869380,-87.655500,41.87,-87.67,member
6037964,DAF0BD6AFE41DFE7,electric_bike,2026-07-31 16:38:46.825,2026-07-31 16:51:12.350,Wells St & Randolph St,CHI00266,NaN,NaN,41.884295,-87.633963,41.88,-87.64,casual
6037965,CAE5402C8F0ED873,electric_bike,2026-07-29 09:51:38.810,2026-07-29 09:59:39.356,Sheffield Ave & Kingsbury St,CHI02023,NaN,NaN,41.910522,-87.653106,41.89,-87.64,member
6037966,8883CB153A66EEAD,electric_bike,2026-07-05 16:42:01.513,2026-07-05 21:38:45.693,Lincoln Ave & Waveland Ave,CHI00446,NaN,NaN,41.948797,-87.675278,41.96,-87.71,member


### Data Cleaning

In [4]:
df.shape

(6037968, 13)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6037968 entries, 0 to 6037967
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 598.9+ MB


In [6]:
# Drop start lat, start lng, end lat, end lng columns
df.drop(columns = ['start_lat', 'start_lng', 'end_lat', 'end_lng'], inplace = True)

In [7]:
# Inspect duplicates
df.duplicated().sum()

np.int64(35)

In [8]:
df['ride_id'].duplicated().sum()

np.int64(35)

In [9]:
# Drop duplicates
df = df.drop_duplicates()

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
# Inspect missing values
df.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1273194
start_station_id      1273194
end_station_name      1336763
end_station_id        1336763
member_casual               0
dtype: int64

In [12]:
df['member_casual'].value_counts()

member_casual
member    3886892
casual    2151041
Name: count, dtype: int64

In [13]:
df['rideable_type'].value_counts()

rideable_type
electric_bike    4146715
classic_bike     1891218
Name: count, dtype: int64

In [14]:
df['started_at'].min()

'2025-07-30 23:35:27.719'

In [15]:
df['started_at'].max()

'2026-07-31 23:58:05.898'

In [16]:
df['ended_at'].min()

'2025-08-01 00:00:07.078'

In [17]:
df['ended_at'].max()

'2026-07-31 23:59:52.627'

In [18]:
warnings.filterwarnings('ignore')

In [19]:
# Convert object to datetime
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

In [20]:
df[['started_at', 'ended_at']].dtypes

started_at    datetime64[ns]
ended_at      datetime64[ns]
dtype: object

In [21]:
# Add ride length column
df['ride_length_mins'] = round((df['ended_at'] - df['started_at']).dt.total_seconds() / 60,2)

In [22]:
df['ride_length_mins'].head()

0    15.13
1    18.34
2    10.97
3     6.62
4    11.54
Name: ride_length_mins, dtype: float64

In [23]:
# Inspect Negative ride lengths
(df['ride_length_mins'] <= 0).sum()

np.int64(156)

In [24]:
# Remove negative ride length rides
df = df[df['ride_length_mins'] > 0]

In [25]:
# Investigate the ride-duration distribution
df['ride_length_mins'].quantile(
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]
)

0.01       0.24
0.25       5.35
0.50       9.34
0.75      16.36
0.95      38.62
0.99      85.09
1.00    1559.95
Name: ride_length_mins, dtype: float64

### Feature Engineering

In [26]:
# Add month, month name, day name and hour columns
df['month'] = df['started_at'].dt.month
df['month_name'] = df['started_at'].dt.month_name()
df['day_of_week'] = df['started_at'].dt.day_name()
df['hour'] = df['started_at'].dt.hour

In [27]:
# Add season column
df['season'] = np.select(
    [
     df['month'].isin([3,4,5]),
     df['month'].isin([6,7,8]),
     df['month'].isin([9,10,11]),
     df['month'].isin([12,1,2])
    ],
    [
     'Spring',
     'Summer',
     'Autumn',
     'Winter'
    ], default = 'Unknown'    
)

In [28]:
# Add day type
df['day_type'] = np.where(
    df['day_of_week'].isin(['Saturday', 'Sunday']),
    'Weekend',
    'Weekday'
)

### MySQL Connection

In [30]:
import pymysql
from sqlalchemy import create_engine

# MySQL Connection
username = 'root'
password = '1234567890'
host = 'localhost'
port = '3306'
database = 'cyclistic_bike'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}')

# Write database to MySQL
table = 'cyclistic_bike_cleaned_dataset'
df.to_sql(table, engine, if_exists = 'replace', index = False)

# Read back sample
pd.read_sql('SELECT * FROM cyclistic_bike_cleaned_dataset LIMIT 5;', engine)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,month,month_name,day_of_week,hour,season,day_type
0,5C00A6B5A60E4AFC,electric_bike,2025-08-28 15:56:47,2025-08-28 16:11:55,None,None,None,None,casual,15.13,8,August,Thursday,15,Summer,Weekday
1,FB25D4B923656D01,electric_bike,2025-08-27 19:15:53,2025-08-27 19:34:14,None,None,None,None,casual,18.34,8,August,Wednesday,19,Summer,Weekday
2,C665B0EC937C860C,electric_bike,2025-08-27 20:27:31,2025-08-27 20:38:30,None,None,None,None,casual,10.97,8,August,Wednesday,20,Summer,Weekday
3,C68C54D5FD9717C2,electric_bike,2025-08-28 15:33:21,2025-08-28 15:39:58,None,None,None,None,casual,6.62,8,August,Thursday,15,Summer,Weekday
4,902D69C29B0E65F8,electric_bike,2025-08-28 09:59:45,2025-08-28 10:11:17,None,None,None,None,casual,11.54,8,August,Thursday,9,Summer,Weekday
